# Error Budget: Autoencoder Latent Dimension $\to$ Waveform Mismatch

Analogous to the PCA error budget, but using a nonlinear autoencoder
for dimensionality reduction of the dephasing curves.

**Autoencoder trained in log(d$\Phi$) space** — the nonlinear encoder/decoder
may handle the dynamic range issue that causes log-space PCA to plateau.
Mismatch is always computed in physical space.

1. **Latent dim sweep**: For each latent dimension L, train an autoencoder,
   reconstruct test curves, compute mismatch
2. **Comparison**: Overlay PCA results (both physical and log space) for reference
3. **Noise sweep**: Add Gaussian noise to the latent codes at varying levels

In [ ]:
import os, glob, re
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("device:", DEVICE)

# ---------- Constants ----------
G    = 6.67430e-11
C    = 2.99792458e8
MSUN = 1.98892e30
YR   = 365.25 * 86400.0
TAU_YR = 4.0

# ---------- Load data ----------
DATA_DIR = "data"
files = sorted(glob.glob(os.path.join(DATA_DIR, "dephasing_sample_*.csv")),
               key=lambda p: int(re.search(r"_(\d+)\.csv$", p).group(1)))

def parse_file(path):
    with open(path) as fh:
        lines = fh.readlines()
    params = {}
    for item in lines[0].strip().split(","):
        k, v = item.split("=")
        params[k.strip()] = float(v)
    data = np.loadtxt(lines[2:], delimiter=",")
    return params, data[:, 0], data[:, 1]

m1  = np.empty(len(files))
m2  = np.empty(len(files))
rho = np.empty(len(files))
gsp = np.empty(len(files))
fs, dphis = [], []
for i, fp in enumerate(files):
    p, f, d = parse_file(fp)
    m1[i], m2[i], rho[i], gsp[i] = p["m1"], p["m2"], p["rhosp"], p["gsp"]
    fs.append(f); dphis.append(d)

# ---------- Frequency normalization ----------
def f_isco(m1_msun, m2_msun):
    M = (m1_msun + m2_msun) * MSUN
    return C**3 / (6**1.5 * np.pi * G * M)

def f_tau(m1_msun, m2_msun, tau_sec=TAU_YR * YR):
    Mc = (m1_msun * m2_msun)**(3/5) / (m1_msun + m2_msun)**(1/5) * MSUN
    return (1.0 / np.pi) * (5.0 / (256.0 * tau_sec))**(3/8) * (G * Mc / C**3)**(-5/8)

fc        = f_isco(m1, m2)
f_tau_arr = f_tau(m1, m2)

xs, phys_dphis = [], []
for i in range(len(files)):
    x = (fs[i][:-1] - fc[i]) / (f_tau_arr[i] - fc[i])
    xs.append(x)
    phys_dphis.append(dphis[i][:-1])

xmins = np.array([x.min() for x in xs])
xmaxs = np.array([x.max() for x in xs])
X_LO  = float(np.max(xmins))
X_HI  = float(np.min(xmaxs))
N_X   = 200
x_grid = np.linspace(X_LO, X_HI, N_X)

# Physical dPhi matrix (for mismatch computation)
Y_phys = np.empty((len(files), N_X))
for i in range(len(files)):
    xi = xs[i][::-1]
    yi = phys_dphis[i][::-1]
    Y_phys[i] = np.interp(x_grid, xi, yi)

# Log dPhi matrix (for AE training)
Y_log = np.log(Y_phys)

# ---------- Train / test split (same seed as other notebooks) ----------
idx = np.arange(len(files))
idx_trainval, idx_test = train_test_split(idx, test_size=0.10, random_state=0)
idx_train, idx_val     = train_test_split(idx_trainval, test_size=1/9, random_state=0)
n_test = len(idx_test)

# Center log curves using training mean (for AE)
Y_log_mean = Y_log[idx_train].mean(0)
Y_log_c = Y_log - Y_log_mean

print(f"Loaded {len(files)} systems, test set: {n_test}")
print(f"Y_phys range: [{Y_phys.min():.2e}, {Y_phys.max():.2e}] rad")
print(f"Y_log range: [{Y_log.min():.2e}, {Y_log.max():.2e}]")

In [ ]:
# LISA noise and waveform functions (from Eval notebook)
L      = 2.5e9
f_star = C / (2 * np.pi * L)
M_sun  = 1.98847e30

def P_OMS(f):
    return (1.5e-11)**2 * (1 + (2e-3 / f)**4)

def P_acc(f):
    return (3e-15)**2 * (1 + (0.4e-3 / f)**2) * (1 + (f / 8e-3)**4)

def S_n(f):
    return ((10 / (3 * L**2)) *
            (P_OMS(f) + 2 * (1 + np.cos(f / f_star)**2) * P_acc(f) / (2 * np.pi * f)**4) *
            (1 + (6/10) * (f / f_star)**2))

def phi_vacuum(f, chirp_mass):
    a_V = (C**3) / (np.pi * G * chirp_mass * f)
    return (1/16) * a_V**(5/3)

def phi_ddot_vacuum(f, chirp_mass):
    a_V = (np.pi * G * chirp_mass / C**3)**(-5/3)
    return (12 * np.pi**2 * f**(11/3)) / (5 * a_V)

def h0(f, chirp_mass, phi_dd):
    prefactor = 0.5 * (4 * np.pi**(2/3) * G**(5/3) * chirp_mass**(5/3) * f**(2/3)) / C**4
    return prefactor * np.sqrt(2 * np.pi / phi_dd)

def build_waveform(f, delta_phi, m1_msun, m2_msun):
    m1_kg = m1_msun * M_sun
    m2_kg = m2_msun * M_sun
    chirp_mass = (m1_kg * m2_kg)**(3/5) / (m1_kg + m2_kg)**(1/5)
    phi_v  = phi_vacuum(f, chirp_mass)
    phi_dd = phi_ddot_vacuum(f, chirp_mass)
    return h0(f, chirp_mass, phi_dd) * np.exp(1j * (phi_v - delta_phi))

def inner_product(a, b, f):
    return 4 * np.real(np.trapezoid(np.conj(a) * b / S_n(f), f))

def overlap_maximized(h1, h2, f):
    df = f[1] - f[0]
    z  = 4 * np.fft.ifft(np.conj(h1) * h2 / S_n(f)) * len(f) * df
    return np.max(np.abs(z))

def resample_waveform_data(f, dphi, n_points=4000):
    f_uniform = np.linspace(f.min(), f.max(), n_points)
    return f_uniform, interp1d(f, dphi, kind="cubic")(f_uniform)

SNR_BENCHMARK = 200
N_PARAMS = 4
M_c = N_PARAMS / (2 * SNR_BENCHMARK**2)
print(f"Critical mismatch M_c = {M_c:.2e}")

In [ ]:
print("Precomputing truth waveforms for test set...")
precomp = []
for j in range(n_test):
    sys_i = idx_test[j]
    f_axis = x_grid * (f_tau_arr[sys_i] - fc[sys_i]) + fc[sys_i]
    dphi_true = Y_phys[sys_i]
    f_dense, dphi_true_dense = resample_waveform_data(f_axis, dphi_true)
    h_true = build_waveform(f_dense, dphi_true_dense, m1[sys_i], m2[sys_i])
    dd = np.real(inner_product(h_true, h_true, f_dense))
    precomp.append({
        'f_axis': f_axis,
        'f_dense': f_dense,
        'h_true': h_true,
        'dd': dd,
        'm1': m1[sys_i],
        'm2': m2[sys_i],
    })
print("Done.")

def fast_mismatch(dphi_pred, j):
    p = precomp[j]
    dphi_pred_dense = interp1d(p['f_axis'], dphi_pred, kind='cubic')(p['f_dense'])
    h_pred = build_waveform(p['f_dense'], dphi_pred_dense, p['m1'], p['m2'])
    hd = overlap_maximized(h_pred, p['h_true'], p['f_dense'])
    hh = np.real(inner_product(h_pred, h_pred, p['f_dense']))
    F = np.clip(hd / np.sqrt(hh * p['dd']), 0.0, 1.0)
    return 1 - F

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, d_in=N_X, d_latent=4, width=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, width),  nn.ReLU(),
            nn.Linear(width, width // 2), nn.ReLU(),
            nn.Linear(width // 2, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, width // 2), nn.ReLU(),
            nn.Linear(width // 2, width),    nn.ReLU(),
            nn.Linear(width, d_in),
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def to_t(a):
    return torch.as_tensor(a, dtype=torch.float32, device=DEVICE)

AE_EPOCHS = 2000
AE_BATCH  = 256
AE_LR     = 1e-3

def train_autoencoder(latent_dim):
    ae = Autoencoder(d_in=N_X, d_latent=latent_dim).to(DEVICE)
    optimizer = torch.optim.Adam(ae.parameters(), lr=AE_LR)
    loss_fn = nn.MSELoss()

    Y_tr_t = to_t(Y_log_c[idx_train])
    Y_va_t = to_t(Y_log_c[idx_val])
    n_tr = Y_tr_t.shape[0]

    best_val = float('inf')
    best_state = None

    for epoch in range(AE_EPOCHS):
        ae.train()
        perm = torch.randperm(n_tr, device=DEVICE)
        for s in range(0, n_tr, AE_BATCH):
            b = perm[s:s + AE_BATCH]
            optimizer.zero_grad()
            recon, _ = ae(Y_tr_t[b])
            loss = loss_fn(recon, Y_tr_t[b])
            loss.backward()
            optimizer.step()

        ae.eval()
        with torch.no_grad():
            recon_va, _ = ae(Y_va_t)
            val_loss = loss_fn(recon_va, Y_va_t).item()

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = {k: v.detach().clone() for k, v in ae.state_dict().items()}


        if epoch % 200 == 0:
            print(f"    epoch {epoch:4d}  val {val_loss:.4e}")

    ae.load_state_dict(best_state)
    return ae, best_val

print(f"AE config: epochs={AE_EPOCHS}, batch={AE_BATCH}, lr={AE_LR}, patience={PATIENCE}")

# Part 1: AE Reconstruction Mismatch vs Latent Dimension

With *perfect* latent recovery, how many AE latent dimensions are needed
for the reconstructed dephasing to produce faithful waveforms (mismatch $< M_c$)?

For each latent dimension L, we train a fresh autoencoder from scratch,
encode the test curves, decode, and compute mismatch against truth.

In [ ]:
LATENT_DIMS = [2, 4, 6, 8, 10, 12, 16, 20, 24, 32, 40, 48, 56, 64, 72, 80]

ae_models = {}
mismatch_vs_L = {}
ae_latent_codes = {}
ae_latent_stds = {}

for L in LATENT_DIMS:
    print()
    print("=" * 60)
    print(f"Latent dim L = {L}")
    print("=" * 60)

    torch.manual_seed(0)
    np.random.seed(0)

    ae, best_val = train_autoencoder(L)
    print(f"  Best val MSE: {best_val:.4e}")

    # Encode + decode test curves -> physical dPhi
    ae.eval()
    with torch.no_grad():
        recon_c, z_test = ae(to_t(Y_log_c[idx_test]))
        _, z_train = ae(to_t(Y_log_c[idx_train]))

    Y_recon_phys = np.exp(recon_c.cpu().numpy() + Y_log_mean)
    z_test_np = z_test.cpu().numpy()
    z_std = z_train.cpu().numpy().std(axis=0)

    # Compute mismatch for each test system
    mismatches = np.empty(n_test)
    for j in range(n_test):
        mismatches[j] = fast_mismatch(Y_recon_phys[j], j)

    ae_models[L] = ae
    mismatch_vs_L[L] = mismatches
    ae_latent_codes[L] = z_test_np
    ae_latent_stds[L] = z_std

    med = np.median(mismatches)
    frac = np.mean(mismatches > M_c)
    p95 = np.percentile(mismatches, 95)
    print(f"  median M={med:.3e}  p95={p95:.3e}  frac>Mc={frac:.3f}")

print()
print("Latent dim sweep complete.")

In [ ]:
print("Computing PCA comparison at same latent dims...")

MAX_L = max(LATENT_DIMS)

# Physical-space PCA
pca_phys = PCA(n_components=MAX_L).fit(Y_phys[idx_train])
coeffs_phys = pca_phys.transform(Y_phys[idx_test])

# Log-space PCA
pca_log = PCA(n_components=MAX_L).fit(Y_log[idx_train])
coeffs_log = pca_log.transform(Y_log[idx_test])

mismatch_pca_phys = {}
mismatch_pca_log = {}

for L in LATENT_DIMS:
    # Physical PCA: reconstruct directly in physical space
    Y_recon_pp = coeffs_phys[:, :L] @ pca_phys.components_[:L] + pca_phys.mean_
    mm_pp = np.empty(n_test)
    for j in range(n_test):
        mm_pp[j] = fast_mismatch(Y_recon_pp[j], j)
    mismatch_pca_phys[L] = mm_pp

    # Log PCA: reconstruct in log space, exp to physical
    Y_recon_lp = coeffs_log[:, :L] @ pca_log.components_[:L] + pca_log.mean_
    Y_recon_lp_phys = np.exp(Y_recon_lp)
    mm_lp = np.empty(n_test)
    for j in range(n_test):
        mm_lp[j] = fast_mismatch(Y_recon_lp_phys[j], j)
    mismatch_pca_log[L] = mm_lp

    frac_ae = np.mean(mismatch_vs_L[L] > M_c)
    frac_pp = np.mean(mm_pp > M_c)
    frac_lp = np.mean(mm_lp > M_c)
    print(f"  L={L:3d}  AE={frac_ae:.3f}  PCA(phys)={frac_pp:.3f}  PCA(log)={frac_lp:.3f}")

print("Done.")

In [ ]:
Ls = np.array(LATENT_DIMS)
fracs_ae   = np.array([np.mean(mismatch_vs_L[L] > M_c) for L in LATENT_DIMS])
fracs_pp   = np.array([np.mean(mismatch_pca_phys[L] > M_c) for L in LATENT_DIMS])
fracs_lp   = np.array([np.mean(mismatch_pca_log[L] > M_c) for L in LATENT_DIMS])

medians_ae = np.array([np.median(mismatch_vs_L[L]) for L in LATENT_DIMS])
medians_pp = np.array([np.median(mismatch_pca_phys[L]) for L in LATENT_DIMS])
medians_lp = np.array([np.median(mismatch_pca_log[L]) for L in LATENT_DIMS])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: median mismatch
ax1.plot(Ls, medians_ae, 'o-', color='C2', lw=2, ms=5, label='Autoencoder (log space)')
ax1.plot(Ls, medians_pp, 's-', color='C0', lw=1.5, ms=4, label='PCA (physical space)')
ax1.plot(Ls, medians_lp, '^-', color='C1', lw=1.5, ms=4, label='PCA (log space)')
ax1.axhline(M_c, color='red', ls='--', lw=1.5, label=f'$M_c = {M_c:.0e}$')
ax1.set(xlabel='Latent / PCA dimension', ylabel='Median mismatch',
        yscale='log', title='Median Reconstruction Mismatch')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: fraction exceeding M_c
ax2.plot(Ls, fracs_ae, 'o-', color='C2', lw=2, ms=5, label='Autoencoder (log space)')
ax2.plot(Ls, fracs_pp, 's-', color='C0', lw=1.5, ms=4, label='PCA (physical space)')
ax2.plot(Ls, fracs_lp, '^-', color='C1', lw=1.5, ms=4, label='PCA (log space)')
ax2.set(xlabel='Latent / PCA dimension',
        ylabel=r'Fraction with $M > M_c$',
        title='Fraction Exceeding Critical Mismatch',
        ylim=(-0.02, 1.02))
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Report optimal L
L_opt = None
for L in LATENT_DIMS:
    if np.mean(mismatch_vs_L[L] > M_c) == 0:
        L_opt = L
        print(f"Minimum AE latent dim for 100% below M_c: L={L_opt}")
        break
if L_opt is None:
    for thresh in [0.01, 0.05, 0.10]:
        for L in LATENT_DIMS:
            if np.mean(mismatch_vs_L[L] > M_c) <= thresh:
                print(f"First L with <= {thresh*100:.0f}% exceeding M_c: L={L} ({np.mean(mismatch_vs_L[L] > M_c)*100:.1f}%)")
                if L_opt is None:
                    L_opt = L
                break
    if L_opt is None:
        L_opt = LATENT_DIMS[-1]
        print(f"Warning: even L={L_opt} has {np.mean(mismatch_vs_L[L_opt] > M_c)*100:.1f}% exceeding M_c")

# Part 2: Latent Noise Sensitivity

For a given latent dimension L, how much noise in the AE latent codes can be tolerated?

**Noise model**: add i.i.d. Gaussian noise to the latent codes, scaled by the
training-set standard deviation of each latent dimension. The noise std $\sigma_n$
in normalized space directly corresponds to the MLP's per-component prediction RMSE.

In [ ]:
# Select latent dims for noise analysis
L_NOISE = sorted(set([L for L in [4, 24, 48, 80] if L in LATENT_DIMS] + [L_opt]))
print(f"Noise analysis at L = {L_NOISE}")

sigma_ns = np.logspace(-5, -0.3, 30)
noise_results = {}

np.random.seed(42)

for L in L_NOISE:
    print()
    print("=" * 60)
    print(f"L = {L}")
    print("=" * 60)

    ae = ae_models[L]
    z_test_np = ae_latent_codes[L]
    z_std = ae_latent_stds[L]

    mismatch_noise = np.full((len(sigma_ns), n_test), np.nan)

    for si, sigma_n in enumerate(sigma_ns):
        noise = np.random.randn(n_test, L) * sigma_n * z_std
        z_noisy = z_test_np + noise

        # Decode noisy latents -> physical dPhi
        ae.eval()
        with torch.no_grad():
            recon_c = ae.decoder(to_t(z_noisy)).cpu().numpy()
        Y_recon_phys = np.exp(recon_c + Y_log_mean)

        for j in range(n_test):
            mismatch_noise[si, j] = fast_mismatch(Y_recon_phys[j], j)

        med = np.median(mismatch_noise[si])
        frac = np.mean(mismatch_noise[si] > M_c)
        print(f"  sigma_n={sigma_n:.3e}  median M={med:.3e}  frac>Mc={frac:.3f}")

    noise_results[L] = mismatch_noise

In [ ]:
fig, axes = plt.subplots(1, len(L_NOISE), figsize=(7 * len(L_NOISE), 5), squeeze=False)

for col, L in enumerate(L_NOISE):
    ax = axes[0, col]
    mm = noise_results[L]

    med = np.array([np.median(mm[si]) for si in range(len(sigma_ns))])
    p5  = np.array([np.percentile(mm[si], 5) for si in range(len(sigma_ns))])
    p95 = np.array([np.percentile(mm[si], 95) for si in range(len(sigma_ns))])

    ax.fill_between(sigma_ns, p5, p95, alpha=0.2, color='C0', label='5th-95th pctl')
    ax.plot(sigma_ns, med, '-', color='C0', lw=2, label='median')
    ax.axhline(M_c, color='red', ls='--', lw=1.5, label=f'$M_c = {M_c:.0e}$')
    ax.set(xlabel=r'Latent noise $\sigma_n$ (normalized)', ylabel='Mismatch',
           xscale='log', yscale='log', title=f'Noise sensitivity (L={L})')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(L_NOISE), figsize=(7 * len(L_NOISE), 5), squeeze=False)

for col, L in enumerate(L_NOISE):
    ax = axes[0, col]
    mm = noise_results[L]

    fracs_noise = np.array([np.mean(mm[si] > M_c) for si in range(len(sigma_ns))])

    ax.plot(sigma_ns, fracs_noise, 's-', color='C1', ms=4)
    ax.set(xlabel=r'Latent noise $\sigma_n$ (normalized)',
           ylabel=r'Fraction with $M > M_c$',
           xscale='log', title=f'Fraction exceeding $M_c$ (L={L})',
           ylim=(-0.02, 1.02))
    ax.grid(alpha=0.3)

    for si in range(len(sigma_ns) - 1, -1, -1):
        if fracs_noise[si] <= 0.01:
            print(f"L={L}: max sigma_n for <=1% exceeding M_c: {sigma_ns[si]:.3e}")
            break
    else:
        print(f"L={L}: even sigma_n={sigma_ns[0]:.3e} has {fracs_noise[0]*100:.1f}% exceeding M_c")

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("ERROR BUDGET SUMMARY (AUTOENCODER)")
print("=" * 70)

print()
print("1. RECONSTRUCTION MISMATCH (perfect latent recovery)")
print("-" * 50)
for L in LATENT_DIMS:
    frac = np.mean(mismatch_vs_L[L] > M_c)
    status = "PASS" if frac == 0 else f"{frac*100:.1f}% exceed M_c"
    print(f"   L={L:3d}  median M={np.median(mismatch_vs_L[L]):.2e}  {status}")

print()
print("2. LATENT NOISE TOLERANCE")
print("-" * 50)
for L in L_NOISE:
    mm = noise_results[L]
    fracs_noise = np.array([np.mean(mm[si] > M_c) for si in range(len(sigma_ns))])
    sigma_crit = None
    for si in range(len(sigma_ns) - 1, -1, -1):
        if fracs_noise[si] <= 0.01:
            sigma_crit = sigma_ns[si]
            break
    if sigma_crit is not None:
        print(f"   L={L}: max sigma_n for <=1% exceeding M_c: {sigma_crit:.3e}")
    else:
        print(f"   L={L}: reconstruction floor too high; noise tolerance near zero")

print()
print("3. COMPARISON WITH PCA")
print("-" * 50)
header = f"   {'L':>4s}  {'AE':>8s}  {'PCA(phys)':>10s}  {'PCA(log)':>10s}"
print(header)
for L in LATENT_DIMS:
    frac_ae = np.mean(mismatch_vs_L[L] > M_c)
    frac_pp = np.mean(mismatch_pca_phys[L] > M_c)
    frac_lp = np.mean(mismatch_pca_log[L] > M_c)
    print(f"   {L:4d}  {frac_ae:8.3f}  {frac_pp:10.3f}  {frac_lp:10.3f}")
print("=" * 70)